# FatFormer-XLA 336 — Train → calibration → master audit

Chạy theo thứ tự trên **Colab GPU**. Source lấy từ Git nhánh **test** và chốt commit.
Đây là lượt thực nghiệm mới, chưa có kết quả train/audit được điền sẵn.

- 336×336; ViT-L/14, nội suy mã hóa vị trí, SRM khớp lưới 24×24.
- Giữ loss, seed 42, 5 epoch và phương pháp ghép cặp. Batch 4 × tích lũy 8 = 32.
- Giữ chuẩn hóa ImageNet và resize vuông trong train/validation như cấu hình train cũ.
  Hiệu chuẩn cuối và audit dùng CLIP resize cạnh ngắn/crop 336, tương ứng giao thức triển khai cũ tại độ phân giải mới.
- Hiệu chuẩn baseline 336 trên 400 ảnh CFE calibration trước train để đóng băng điểm vận hành chọn epoch.
  Hiệu chuẩn lại checkpoint sau train trên đúng 400 ảnh đó; 1.600 ảnh CFE evaluation không dùng để fit.
- 8 archive, 37 job, không giới hạn số ảnh. Train/validation forward là diagnostic.
- Không tự thay mô hình demo 224 đã chốt. Chỉ cân nhắc bản 336 sau khi có kết quả và giới hạn rõ ràng.
- Giữ `RUN_NAME` để resume epoch hoàn tất; dùng tên mới khi đổi cấu hình. Mỗi lần audit có thư mục riêng.

Kho Drive cần giữ cấu trúc `Fatformer/pretrained`, `Fatformer/phase7`, `Fatformer/datasets` như lượt cũ.
Nếu GPU hết bộ nhớ, dùng **run mới**, batch 2 và tích lũy 16; không đổi batch giữa một run.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, os, json, shutil, hashlib, tarfile, gc

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/Fatformer')
RUN_NAME = 'anchored_336_run1'
CODE_REF = None  # None: previous run's pinned commit, otherwise latest test. Or set an exact fixed SHA.
REPO = Path('/content/fatformer-xla-336')
RUNTIME = Path('/content/phase7_runtime336')
STAGE = Path('/content/dataset_local/phase7_336')
RUN_DIR = RUNTIME / 'runs' / RUN_NAME
BACKUP = DRIVE / 'checkpoint' / 'phase7' / RUN_NAME
RUNTIME.mkdir(parents=True, exist_ok=True)

def run(args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd or (REPO if REPO.exists() else '/content'), check=True)

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

run(['nvidia-smi'])
import torch
assert torch.cuda.is_available(), 'A CUDA runtime is required. CPU is not a training preflight PASS.'
free, total = torch.cuda.mem_get_info()
assert free >= 14 * 1024**3, f'Need >=14 GiB FREE VRAM; got {free / 1024**3:.2f}'
print(torch.cuda.get_device_name(), 'free GiB:', free / 1024**3)
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
INFERENCE_BATCH = 4
assert BATCH_SIZE * GRAD_ACCUM_STEPS == 32

## 2. Lấy source từ Git và chốt commit

In [ ]:
saved_ref = BACKUP / 'source_commit.txt'
ref = CODE_REF or (saved_ref.read_text().strip() if saved_ref.exists() else 'test')
if not REPO.exists():
    run(['git', 'clone', '-b', 'test', 'https://github.com/LeeVHoangtk3/fatformer-xla.git', REPO], cwd='/content')
dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO, text=True)
assert not dirty.strip(), 'Tracked code has local edits; preserve them before changing the checkout.'
run(['git', 'fetch', 'origin', 'test'])
run(['git', 'fetch', 'origin', ref])
run(['git', 'merge-base', '--is-ancestor', 'FETCH_HEAD', 'refs/remotes/origin/test'])
run(['git', 'checkout', '--detach', 'FETCH_HEAD'])
SOURCE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'tqdm', 'pyyaml', 'timm', 'einops', 'pytest', 'numpy', 'Pillow', 'scikit-learn', 'pandas', 'matplotlib'])
help_text = subprocess.check_output([sys.executable, '-m', 'src.training.train_crossdomain', '--help'], cwd=REPO, text=True)
assert all(flag in help_text for flag in ['--resume_from', '--benchmark_steps', '--data_root']), 'Stale code: checkout a commit containing the Task 7.11 fixes first.'
import yaml
print('Pinned source:', SOURCE_COMMIT)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from tools.phase7_git_source import git_source_identity
SOURCE_PATHS = ['src', 'tools', 'tests', 'configs', 'DATASET/manifests',
                'notebooks/train_336_master_audit_colab.ipynb',
                'DATASET/logs/phase7/master_audit/runs/master_audit_20261115_040751_7beca5/raw/benchmark_summary.json']
SOURCE_PROOF = git_source_identity(REPO, source_paths=SOURCE_PATHS, expected_branch='test')
assert SOURCE_PROOF['commit'] == SOURCE_COMMIT

## 3. Kiểm checksum kho train và dựng cấu hình 336 riêng

In [ ]:
archive = DRIVE / 'phase7' / 'train_crossdomain_v2.tar'
checksum = Path(str(archive) + '.sha256')
assert archive.is_file() and checksum.is_file(), 'Need both the audited archive and its original .sha256 sidecar.'
expected = checksum.read_text().split()[0]
assert len(expected) == 64 and all(c in '0123456789abcdef' for c in expected.lower())
STAGE.mkdir(parents=True, exist_ok=True)
local_archive = STAGE / archive.name
if not local_archive.exists() or digest(local_archive) != expected:
    shutil.copy2(archive, local_archive)
assert digest(local_archive) == expected, 'Archive SHA256 mismatch; do not extract or train.'
marker = STAGE / '.extracted_sha256'
if not marker.exists() or marker.read_text().strip() != expected:
    assert sys.version_info >= (3, 12), 'Use a runtime with tarfile data extraction filters.'
    with tarfile.open(local_archive) as bundle:
        bundle.extractall(STAGE, filter='data')
    marker.write_text(expected)

assets = RUNTIME / 'pretrained'
assets.mkdir(exist_ok=True)
for name in ('ViT-L-14.pt', 'fatformer_4class_ckpt.pth'):
    source = DRIVE / 'pretrained' / name
    assert source.is_file(), f'Missing model asset: {source}'
    target = assets / name
    if not target.exists() or digest(target) != digest(source):
        shutil.copy2(source, target)
    assert digest(target) == digest(source)

previous_config = BACKUP / 'run_config.yaml'
base = previous_config if previous_config.exists() else REPO / 'configs/train_anchored_336.yaml'
config = yaml.safe_load(base.read_text())
config['dataset'].update(manifest=str(STAGE / 'DATASET/manifests/data_manifest_v2.json'),
                         isolation_report=str(STAGE / 'DATASET/logs/phase7/data_isolation_report.json'),
                         audit_config=str(STAGE / 'configs/phase7_data_v2.yaml'),
                         data_root=str(STAGE), train_dir='', val_dir='', val_split='val_select',
                         degradation_config=str(REPO / 'configs/degradation_336.yaml'))
config['model'].update(clip_path=str(assets / 'ViT-L-14.pt'),
                       checkpoint_init=str(assets / 'fatformer_4class_ckpt.pth'),
                       teacher_checkpoint=str(assets / 'fatformer_4class_ckpt.pth'))
config['training'].update(device='cuda', batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS, save_dir=str(RUN_DIR), backup_dir=str(BACKUP),
                          history_csv=str(RUN_DIR / 'train_history.csv'),
                          calibration_file=str(REPO / 'DATASET/logs/phase7/calibration/baseline.json'))
config['training'].pop('resume_from', None)
CONFIG = RUNTIME / 'train_runtime.yaml'
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print('mini-batch:', config['training']['batch_size'], 'grad accumulation:', config['training']['grad_accum_steps'])
print('Effective batch:', config['training']['batch_size'] * config['training']['grad_accum_steps'])
assert config['dataset']['img_resolution'] == config['dataset']['crop_resolution'] == 336
assert config['training']['epochs'] == 5
assert config['dataset']['preprocessing'] == config['dataset']['validation_preprocessing'] == 'legacy_imagenet'

## 4. Đóng băng điểm vận hành baseline ở 336 trước train

In [ ]:
from tools.master_data_audit import catalog, manifest_maps, open_inventory, audit_catalog, extract_audited
from tools.calibration_data import sha256_file
from tools.phase7_corrected import write_json

specs = catalog(DRIVE)
missing = [s['path'] for s in specs if not Path(s['path']).is_file()]
if missing:
    raise FileNotFoundError('Missing required archives: ' + '\n'.join(missing))
CFE_STAGE = RUNTIME / 'cfe_calibration_images'
CFE_STAGE.mkdir(exist_ok=True)
CFE_PREP = RUNTIME / 'cfe_input_audit'
CFE_PREP.mkdir(exist_ok=True)
inventory = CFE_PREP / 'inventory.sqlite'
import sqlite3
connection = sqlite3.connect(inventory) if inventory.exists() else open_inventory(inventory)
try:
    spec = next(s for s in specs if s['name'] == 'cfe')
    report = audit_catalog([spec], connection, manifest_maps(REPO), CFE_PREP, reuse_completed=True)[0]
    spec['audited_sha256'] = report['archive_sha256']
    cfe_records = extract_audited(spec, connection, CFE_STAGE)
finally:
    connection.close()
assert len(cfe_records) == 2000
CAL_SPLIT = REPO / 'configs/master_audit/calibration_split.json'
EVAL_SPLIT = REPO / 'configs/master_audit/evaluation_split.json'
SELECTION = RUNTIME / 'selection'
SELECTION.mkdir(exist_ok=True)
selection_file = SELECTION / 'baseline336.json'
saved_selection = BACKUP / 'selection'
selection_marker = saved_selection / 'complete.json'
if selection_marker.is_file():
    frozen_selection = json.loads(selection_marker.read_text())
    for name, expected_sha in frozen_selection.items():
        assert digest(saved_selection / name) == expected_sha, 'Saved selection artifact changed.'
    shutil.copytree(saved_selection, SELECTION, dirs_exist_ok=True)
else:
    run([sys.executable, '-m', 'tools.extract_calibration_logits',
         '--checkpoint', assets / 'fatformer_4class_ckpt.pth', '--clip_path', assets / 'ViT-L-14.pt',
         '--data_dir', CFE_STAGE, '--split_file', CAL_SPLIT, '--evaluation_split', EVAL_SPLIT,
         '--output_npz', SELECTION / 'logits.npz', '--input_resolution', '336',
         '--device', 'cuda', '--precision', 'fp16', '--batch_size', str(INFERENCE_BATCH)])
    run([sys.executable, '-m', 'tools.calibrate_and_select', '--logits_npz', SELECTION / 'logits.npz',
         '--checkpoint', assets / 'fatformer_4class_ckpt.pth', '--calibration_split', CAL_SPLIT,
         '--evaluation_split', EVAL_SPLIT, '--output_dir', SELECTION,
         '--model_name', 'baseline336', '--probability_device', 'cuda'])
    saved_selection.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SELECTION, saved_selection, dirs_exist_ok=True)
    write_json(selection_marker, {name:digest(saved_selection / name) for name in ('baseline336.json', 'logits.npz')})
selection = json.loads(selection_file.read_text())
assert selection['input_resolution'] == 336
assert selection['checkpoint']['sha256'] == digest(assets / 'fatformer_4class_ckpt.pth')
assert selection['partition_validation']['calibration_split_sha256'] == digest(CAL_SPLIT)
assert selection['partition_validation']['evaluation_split_sha256'] == digest(EVAL_SPLIT)
assert selection['extraction_batch_size'] == INFERENCE_BATCH and selection['probability_device'] == 'cuda'
config['training']['calibration_file'] = str(selection_file)
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print('Frozen baseline 336 selection:', selection_file)

## 5. Preflight đầy đủ trên dữ liệu thật và GPU

In [ ]:
PREFLIGHT = RUNTIME / 'preflight_train.json'
run([sys.executable, 'tools/preflight_phase7_train.py', '--config', CONFIG, '--output', PREFLIGHT])
preflight = json.loads(PREFLIGHT.read_text())
assert preflight['ready_to_train'] is True

## 6. Đo 100 mini-batch trước train; không tự giảm số epoch

In [ ]:
RESUMING = (RUN_DIR / 'checkpoint_latest.pth').exists() or (BACKUP / 'checkpoint_latest.pth').exists()
BENCHMARK = RUNTIME / 'benchmark' / 'train_benchmark_100steps.json'
if not RESUMING:
    run([sys.executable, '-m', 'src.training.train_crossdomain', '--config', CONFIG,
         '--benchmark_steps', '100', '--benchmark_warmup', '5', '--output', BENCHMARK,
         '--save_dir', RUNTIME / 'benchmark' / 'scratch'])
    measurement = json.loads(BENCHMARK.read_text())
    assert measurement['evidence_level'] == 'measured' and measurement['device'].startswith('cuda')
    print(json.dumps(measurement, indent=2))

else:
    print('Resume detected: benchmark skipped and the original epoch horizon retained.')

## 7. Huấn luyện 5 epoch hoặc resume từ epoch hoàn tất

In [ ]:
# Recheck code identity and final runtime configuration before training.
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() == SOURCE_COMMIT
run(['git', 'diff', '--exit-code', 'HEAD', '--', 'src', 'configs', 'tools'])
preflight = json.loads(PREFLIGHT.read_text())
if preflight['config_sha256'] != digest(CONFIG) or not preflight['ready_to_train']:
    run([sys.executable, 'tools/preflight_phase7_train.py', '--config', CONFIG, '--output', PREFLIGHT])
assert json.loads(PREFLIGHT.read_text())['ready_to_train'] is True
RUN_DIR.mkdir(parents=True, exist_ok=True)
BACKUP.mkdir(parents=True, exist_ok=True)
latest = RUN_DIR / 'checkpoint_latest.pth'
remote = BACKUP / 'checkpoint_latest.pth'
if not latest.exists() and not remote.exists():
    assert BENCHMARK.exists(), 'Run the measured benchmark successfully before starting a new run.'
    measured = json.loads(BENCHMARK.read_text())
    assert measured['evidence_level'] == 'measured' and measured['gpu_name'] == torch.cuda.get_device_name()
    assert measured['context']['source_sha256'] == json.loads(PREFLIGHT.read_text())['source_sha256']
    for key in ('manifest', 'degradation_config'):
        context_key = 'train_manifest_sha256' if key == 'manifest' else key + '_sha256'
        assert measured['context'][context_key] == digest(config['dataset'][key]), 'Inputs changed; rerun benchmark.'
    assert measured['context']['calibration_file_sha256'] == digest(config['training']['calibration_file'])
    for section, keys in [('training', ('batch_size', 'grad_accum_steps', 'use_amp', 'learning_rate', 'anchor_learning_rate')),
                          ('loss', ('weight_ce', 'weight_ce_deg', 'alpha_kd', 'beta_cons', 'weight_feat', 'weight_guard', 'temperature'))]:
        for key in keys:
            assert measured['context'][key] == config[section][key], 'Hyperparameters changed; rerun benchmark.'
if not latest.exists() and remote.exists():
    remote_checksum = Path(str(remote) + '.sha256')
    assert remote_checksum.exists(), 'Legacy/unverified latest: use a new RUN_NAME or a full verified resume state.'
    temporary = Path(str(latest) + '.download')
    shutil.copy2(remote, temporary)
    assert digest(temporary) == remote_checksum.read_text().strip(), 'Resume checksum mismatch.'
    os.replace(temporary, latest)
    for name in (config['training']['save_name'], 'best_effort.pth'):
        if (BACKUP / name).exists():
            shutil.copy2(BACKUP / name, RUN_DIR / name)

(BACKUP / 'run_config.yaml').write_text(CONFIG.read_text())
(BACKUP / 'source_commit.txt').write_text(SOURCE_COMMIT + '\n')
if BENCHMARK.exists():
    shutil.copy2(BENCHMARK, BACKUP / BENCHMARK.name)
shutil.copy2(PREFLIGHT, BACKUP / PREFLIGHT.name)
command = [sys.executable, '-m', 'src.training.train_crossdomain', '--config', CONFIG]
if latest.exists():
    command += ['--resume_from', latest]
run(command)
print('Run process completed. Review the groups/guard gates; this does not automatically pass G7.2.')

## 8. Kiểm tra kết quả train và đường hội tụ

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
csv_path = RUN_DIR / 'val_groups.csv'
assert csv_path.exists(), 'No validated epoch log was produced.'
df = pd.read_csv(csv_path)
print(df.to_string(index=False))
fig, axes = plt.subplots(2, 1, figsize=(10, 10))
ax = axes[0]
for key in ('clean_r', 'clean_f', 'q30_r', 'q30_f', 'worst'):
    ax.plot(df['epoch'], df[key] * 100, marker='o', label=key)
ax.axhline(90, color='gray', linestyle='--', label='Clean Real requirement')
ax.set(xlabel='Completed epoch', ylabel='Recall / worst group (%)', title='FatFormer-XLA 336')
ax.legend(); ax.grid(alpha=.2)
history = pd.read_csv(RUN_DIR / 'train_history.csv')
for key in ('loss_total', 'loss_ce', 'loss_kd', 'loss_cons', 'weighted_feat', 'weighted_guard'):
    if key in history:
        axes[1].plot(history['epoch'], history[key], marker='o', label=key)
axes[1].set(xlabel='Completed epoch', ylabel='Loss', title='Training loss components')
axes[1].legend(); axes[1].grid(alpha=.2)
fig.tight_layout()
chart = RUN_DIR / 'm7_convergence.png'
fig.savefig(chart, dpi=160, bbox_inches='tight')
plt.show()
shutil.copy2(chart, BACKUP / chart.name)
if not (RUN_DIR / config['training']['save_name']).exists():
    print('No official best is present. best_effort is for analysis only.')
print('Artifacts:', BACKUP)

## 9. Hiệu chuẩn checkpoint mới, đóng gói và kiểm chứng bằng ảnh thật

In [ ]:
import uuid
from datetime import datetime, timezone
FINAL_ID = 'resolution336_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
FINAL = RUNTIME / 'finalized' / FINAL_ID
FINAL.mkdir(parents=True, exist_ok=False)
best = RUN_DIR / config['training']['save_name']
training_rows = list(__import__('csv').DictReader((RUN_DIR / 'train_history.csv').open()))
assert [int(r['epoch']) for r in training_rows] == [1, 2, 3, 4, 5], 'Require five completed epochs before full audit.'
selection_guard_passed = best.is_file()
if not selection_guard_passed:
    best = RUN_DIR / 'best_effort.pth'
    if not best.is_file():
        raise RuntimeError('Training produced no eligible or best-effort candidate to evaluate.')
    print('VALIDATION GUARD FAILED: auditing best_effort diagnostically; this is not an accepted final model.')
raw_metadata = torch.load(best, map_location='cpu', weights_only=True, mmap=True)['metadata']
assert raw_metadata['input_resolution'] == 336
assert raw_metadata['is_best' if selection_guard_passed else 'is_best_effort'] is True
run([sys.executable, '-m', 'tools.extract_calibration_logits', '--checkpoint', best,
     '--clip_path', assets / 'ViT-L-14.pt', '--data_dir', CFE_STAGE, '--split_file', CAL_SPLIT,
     '--evaluation_split', EVAL_SPLIT, '--output_npz', FINAL / 'logits.npz',
     '--device', 'cuda', '--precision', 'fp16', '--batch_size', str(INFERENCE_BATCH)])
run([sys.executable, '-m', 'tools.calibrate_and_select', '--logits_npz', FINAL / 'logits.npz',
     '--checkpoint', best, '--calibration_split', CAL_SPLIT, '--evaluation_split', EVAL_SPLIT,
     '--output_dir', FINAL, '--model_name', 'model336', '--probability_device', 'cuda'])
run([sys.executable, '-m', 'tools.package_unified_fatformer', '--checkpoint', best,
     '--output', FINAL / 'unified_fatformer_336.pth', '--calibration', FINAL / 'model336.json',
     '--metadata_out', FINAL / 'package_metadata.json'])
# Equivalence uses 20 actual calibration images; no evaluation examples are needed here.
calib_paths = [r['path'] for label in (0, 1) for r in
               [r for r in cfe_records if r['split'] == 'calibration' and r['label'] == label][:10]]
assert len(calib_paths) == 20
run([sys.executable, '-m', 'tools.verify_unified_package',
     '--checkpoint', FINAL / 'unified_fatformer_336.pth', '--reference', best,
     '--output', FINAL / 'package_verification.json', '--device', 'cuda', '--precision', 'fp16',
     '--images', *calib_paths])
from tools.phase7_corrected import validate_inputs
proof = validate_inputs(best, FINAL / 'model336.json', FINAL / 'logits.npz', CAL_SPLIT, EVAL_SPLIT,
                        expected_checkpoint=digest(best))
assert proof['extraction_batch_size'] == INFERENCE_BATCH and proof['inference_precision'] == 'fp16'
verification = json.loads((FINAL / 'package_verification.json').read_text())
assert verification['tensors_equal'] and verification['real_image_acceptance']
CONTRACT = dict(schema='resolution336-run/v1', input_resolution=336,
    source_commit=SOURCE_COMMIT, package_sha256=digest(FINAL / 'unified_fatformer_336.pth'),
    source_checkpoint_sha256=digest(best), inference_batch=INFERENCE_BATCH,
    train_batch=BATCH_SIZE, train_grad_accum_steps=GRAD_ACCUM_STEPS,
    calibration_sha256=digest(FINAL / 'model336.json'), verification=verification,
    epoch=raw_metadata['epoch'], selection_split='val_select', selection_guard_passed=selection_guard_passed,
    scientific_status='candidate; acceptance requires fresh master audit')
write_json(FINAL / 'run_contract.json', CONTRACT)
PACKAGE_FOLDER = BACKUP / 'packages' / FINAL_ID
shutil.copytree(FINAL, PACKAGE_FOLDER, dirs_exist_ok=False)
PACKAGE_DRIVE = PACKAGE_FOLDER / 'unified_fatformer_336.pth'
PACKAGE_SHA = CONTRACT['package_sha256']
SOURCE_SHA = CONTRACT['source_checkpoint_sha256']
print('New package:', PACKAGE_DRIVE)

## 10. Master audit: tạo run riêng và lưu bằng chứng khi có lỗi

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from contextlib import contextmanager
import importlib.util, json, os, shutil, subprocess, sys, tempfile, uuid

DRIVE_ROOT = Path('/content/drive/MyDrive/Fatformer')
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(DRIVE_ROOT)
RUN_ID = 'master_audit_336_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
RUN_ROOT = Path('/content')/RUN_ID
OUTPUT = RUN_ROOT/'results'
STAGING = RUN_ROOT/'staging'
for folder in (OUTPUT, STAGING):
    folder.mkdir(parents=True, exist_ok=False)
BATCH_SIZE = INFERENCE_BATCH  # Không tự đổi batch giữa chừng. OOM: run mới, batch nhỏ hơn, ghi khác protocol.
BOOTSTRAP = 1000
FULL_REPOSITORY_TESTS = False  # Mặc định chạy nhóm test liên quan, không gọi là full-suite.
RESUME_AUDIT_FROM = None  # Thư mục results cũ còn trong runtime; chỉ tái sử dụng content audit đã hoàn tất.
state = dict(run_id=RUN_ID, execution_status='configured', steps=[],
    scientific_status='post_hoc_remeasurement; not a new blind test',
    training=True, calibration_refit=True, input_resolution=336, batch_size=BATCH_SIZE,
    bootstrap=BOOTSTRAP, full_repository_tests_requested=FULL_REPOSITORY_TESTS)

def persist(status):
    state['execution_status'] = status
    (OUTPUT/'run_status.json').write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')
    package = Path(shutil.make_archive(str(RUN_ROOT/RUN_ID), 'zip', OUTPUT))
    destination = DRIVE_ROOT/'audit_reports'/RUN_ID
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(package, destination/package.name)
    print('Evidence ZIP:', destination/package.name, '| status:', status)

@contextmanager
def step(name):
    entry = dict(name=name, status='running')
    state['steps'].append(entry)
    try:
        yield
    except Exception as error:
        entry.update(status='failed', error_type=type(error).__name__, error=str(error))
        persist('failed')
        raise
    else:
        entry['status'] = 'completed'
        (OUTPUT/'run_status.json').write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')

print('Run:', RUN_ID)

state['training_contract'] = CONTRACT

In [ ]:
# Source is the same clean, pinned test commit used by training.
with step('git_source'):
    source = git_source_identity(REPO, source_paths=SOURCE_PATHS, expected_branch='test')
    if source != SOURCE_PROOF:
        raise ValueError('Source changed since training.')
    state['git_source'] = source
    write_json(OUTPUT / 'git_source.json', source)
    write_json(OUTPUT / 'training_contract.json', CONTRACT)

In [ ]:
# 3. Giữ torch/torchvision runtime, chỉ cài dependency phụ còn thiếu; lưu môi trường thực tế.
with step('environment_and_tests'):
    dependencies = {'pytest':'pytest', 'yaml':'PyYAML', 'sklearn':'scikit-learn',
        'matplotlib':'matplotlib', 'regex':'regex', 'tqdm':'tqdm', 'scipy':'scipy'}
    missing = [package for module, package in dependencies.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    import torch, torchvision, numpy as np
    from tools.master_data_audit import (catalog, manifest_maps, open_inventory,
        audit_catalog, restore_audit_inventory, check_overlap, evaluate_archive, final_metrics, jobs_for, export_summary_csv)
    from tools.phase7_corrected import write_json
    from tools.calibration_data import sha256_file
    from tools.infer_unified import load_unified_model, benchmark_latency
    if not torch.cuda.is_available():
        raise RuntimeError('Cần runtime GPU. Không tự chuyển CPU cho lượt benchmark này.')
    if BATCH_SIZE < 1 or BOOTSTRAP < 2:
        raise ValueError('Batch phải dương; bootstrap ít nhất 2.')
    torch.set_num_threads(4)
    torch.manual_seed(42)
    np.random.seed(42)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    state['environment'] = dict(python=sys.version, torch=str(torch.__version__), torchvision=str(torchvision.__version__),
        gpu=torch.cuda.get_device_name(0), precision='FP16 backbone autocast; FP32 logits/softmax', tf32=False)
    (OUTPUT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
    test_files = ['tests/test_input_resolution.py', 'tests/test_resolution336_workflow.py', 'tests/test_unified_packaging.py', 'tests/test_master_data_audit.py', 'tests/test_canonical_loader.py',
        'tests/test_calibration_partitions.py', 'tests/test_temperature_convergence.py',
        'tests/test_phase7_corrected.py', 'tests/test_phase7_git_source.py', 'tests/test_phase7_acceptance.py', 'tests/test_router_pipeline.py',
        'tests/test_loss_v2.py', 'tests/test_selection_metric.py', 'tests/test_selection_regressions.py', 'tests/test_degradation_v2.py']
    command = [sys.executable, '-m', 'pytest', *(['tests'] if FULL_REPOSITORY_TESTS else test_files),
               '-q', '-p', 'no:cacheprovider', '--junitxml', str(OUTPUT/'tests.xml')]
    with (OUTPUT/'tests_console.txt').open('w', encoding='utf-8') as handle:
        outcome = subprocess.run(command, cwd=REPO, stdout=handle, stderr=subprocess.STDOUT, text=True)
    state['tests'] = dict(command=command, exit_code=outcome.returncode,
                         scope='full repository' if FULL_REPOSITORY_TESTS else 'targeted related files; not full repository')
    print((OUTPUT/'tests_console.txt').read_text(encoding='utf-8')[-4000:])
    if outcome.returncode:
        raise RuntimeError('Tests không đạt. Dừng; xem tests.xml và console, không bỏ qua để chạy tiếp.')

In [ ]:
# 4. Tiền kiểm đúng cả danh mục. Sửa đường dẫn ở đây nếu Drive đã được tổ chức khác.
with step('catalog_preflight'):
    ARCHIVES = catalog(DRIVE_ROOT)
    missing = [s['path'] for s in ARCHIVES if not Path(s['path']).is_file()]
    if not PACKAGE_DRIVE.is_file():
        missing.append(str(PACKAGE_DRIVE))
    write_json(OUTPUT/'catalog_preflight.json', dict(archives=ARCHIVES, missing=missing))
    if missing:
        raise FileNotFoundError('Thiếu các đầu vào bắt buộc:' + chr(10) + chr(10).join(missing))
    maps = manifest_maps(REPO)
    state['input_manifest_hashes'] = {str(p.relative_to(REPO)):sha256_file(p)
        for folder, pattern in [('DATASET/manifests','*.json'), ('configs/master_audit','*.json')]
        for p in (REPO/folder).glob(pattern)}
    db = open_inventory(OUTPUT/'image_inventory.sqlite')
    if RESUME_AUDIT_FROM:
        state['audit_resume'] = restore_audit_inventory(RESUME_AUDIT_FROM, OUTPUT, db, state['git_source'])
    print('Đủ đường dẫn 8 archives; chưa đồng nghĩa nội dung đã được audit.')

In [ ]:
# 5. Audit toàn bộ ảnh, không giải nén toàn kho. Lỗi ảnh/hash/nhãn/count sẽ dừng.
with step('all_archive_content_audit'):
    audits = audit_catalog(ARCHIVES, db, maps, OUTPUT, reuse_completed=bool(RESUME_AUDIT_FROM))
    state['archive_image_records'] = sum(row['images'] for row in audits)
    state['distinct_encoded_file_hashes'] = db.execute('SELECT count(DISTINCT sha256) FROM images').fetchone()[0]
    write_json(OUTPUT/'archive_audit_summary.json', dict(archives=audits,
        total_records=state['archive_image_records'], unique_encoded_hashes=state['distinct_encoded_file_hashes'],
        note='Archives/splits/replay/degraded views can overlap. Counts do not establish statistical independence.'))
    print('Đã audit bản ghi ảnh:', state['archive_image_records'])

In [ ]:
# 6. Exact-content leakage trước inference. Không giả đã kiểm pHash/generator disjoint toàn kho.
with step('content_overlap'):
    overlap = check_overlap(db, OUTPUT/'content_overlap.json')
    state['perceptual_leakage_audit'] = 'not_run; exact SHA256 only'
    print('Không phát hiện exact-content leakage trong các đối chiếu đã định. pHash chưa chạy.')

In [ ]:
# 7. Nạp đúng package qua canonical loader; không coi state_dict là nn.Module.
with step('model_load'):
    current_source = git_source_identity(REPO, source_paths=SOURCE_PATHS, expected_branch='test')
    if current_source['commit'] != state['git_source']['commit'] or current_source['files'] != state['git_source']['files']:
        raise ValueError('Source/input thay đổi sau bước kiểm tra Git.')
    package = RUN_ROOT/'unified_fatformer_336.pth'
    shutil.copy2(PACKAGE_DRIVE, package)
    if sha256_file(package) != PACKAGE_SHA:
        raise ValueError('Package không khớp model đã chọn.')
    device = torch.device('cuda')
    model, metadata = load_unified_model(str(package), clip_path='', device='cuda')
    if metadata['source_checkpoint_sha256'] != SOURCE_SHA or metadata['layout']['indices'] != [7,15,23] or model.router_enabled:
        raise ValueError('Sai source/layout/router.')
    if metadata.get('input_resolution') != 336 or metadata['calibration'].get('input_resolution') != 336:
        raise ValueError('Package/calibration must declare 336.')
    if sha256_file(PACKAGE_FOLDER / 'model336.json') != CONTRACT['calibration_sha256']:
        raise ValueError('Calibration artifact changed after packaging.')
    state['package_sha256'] = PACKAGE_SHA
    state['source_checkpoint_sha256'] = SOURCE_SHA
    state['temperature'] = model.clean_temperature
    state['threshold'] = model.fixed_threshold
    state['protocol_comparison'] = '336 versus historical 224; FP16/TF32-off, new resolution and inference batch explicitly recorded'
    write_json(OUTPUT/'package_metadata.json', metadata)

In [ ]:
# 8. Chạy model thật trên toàn bộ archive; train/val/calibration là diagnostic riêng.
# Không giới hạn sample. Chỉ một archive giải nén mỗi lần; tự dọn staging khi kết thúc hoặc lỗi.
with step('all_dataset_predictions'):
    reports = []
    expected_jobs = set()
    for spec in ARCHIVES:
        partitions = [dict(split=row[0]) for row in db.execute('SELECT DISTINCT split FROM images WHERE archive=?', (spec['name'],))]
        expected_jobs.update((spec['name'], split, condition) for split, condition, _ in jobs_for(spec, partitions))
    assert len(expected_jobs) == 37, 'Catalog/job contract changed; inspect before claiming full coverage.'
    for spec in ARCHIVES:
        with tempfile.TemporaryDirectory(prefix=spec['name']+'_', dir=STAGING) as temporary:
            rows = evaluate_archive(spec, db, Path(temporary), OUTPUT/'predictions', model, device,
                                    batch_size=BATCH_SIZE, bootstrap=BOOTSTRAP, package_sha256=PACKAGE_SHA)
            reports.extend(rows)
            write_json(OUTPUT/'job_reports.json', reports)
        print('Xong archive:', spec['name'])
    if len(reports) != len(expected_jobs) or {(r['archive'],r['split'],r['condition']) for r in reports} != expected_jobs:
        raise ValueError('Job coverage thiếu hoặc trùng. Không công bố full run.')
    state['expected_jobs'] = len(expected_jobs)
    state['completed_jobs'] = len(reports)
    state['measured_prediction_records'] = sum(r['pooled']['total_samples'] for r in reports)
    summary = final_metrics(reports, OUTPUT/'predictions', model.fixed_threshold, BOOTSTRAP)
    write_json(OUTPUT/'benchmark_summary.json', summary)
    export_summary_csv(summary, OUTPUT/'measured_condition_and_subset_summary.csv')
    print('Lượt dự đoán đo mới:', state['measured_prediction_records'])

In [ ]:
# 9. Latency synthetic riêng: paired backbone/package, warmup20, steps200, median/p95/overhead.
with step('latency_and_gates'):
    latency = benchmark_latency(model, device, batch_sizes=[1,INFERENCE_BATCH], n_warmup=20, n_steps=200,
        output_json=str(OUTPUT/'latency.json'), checkpoint_path=str(package))
    gates = {'G7.0': dict(status='NOT_REEVALUATED', reason='Legacy loader/CI DoD needs a separate audit; tests here do not close it.'),
        'G7.1': dict(status='NOT_REEVALUATED', reason='Historical Gen2 comparison is not a new measurement of this package.'),
        'G7.2': summary['G72'],
        'G7.3': dict(status='PENDING', measured_gpu_overhead=latency['overhead_meets_target'],
            single_backbone_forward_checked=True, best_by_condition_comparison='NOT_RUN')}
    gates['training_selection'] = dict(status='PASS' if CONTRACT['selection_guard_passed'] else 'FAIL',
        reason='Frozen val_select Real guard; best_effort is diagnostic only')
    write_json(OUTPUT/'gates.json', gates)
    print(json.dumps(gates, indent=2, ensure_ascii=False))

In [ ]:
# Compare measured summaries only after all 37 current jobs have completed.
with step('comparison_with_224'):
    historical_path = REPO / 'DATASET/logs/phase7/master_audit/runs/master_audit_20261115_040751_7beca5/raw/benchmark_summary.json'
    historical = json.loads(historical_path.read_text())
    comparison_rows = []
    for scope in ('cfe', 'indist_core18'):
        for condition, current in summary[scope].items():
            old = historical[scope][condition]
            assert current['pooled']['total_samples'] == old['pooled']['total_samples']
            for metric in ('accuracy', 'real_acc', 'fake_acc', 'auc', 'fake_at_real90'):
                before = old['pooled'][metric]
                after = current['pooled'][metric]
                comparison_rows.append(dict(scope=scope, condition=condition, metric=metric,
                    reference_224=before, candidate_336=after, delta_percentage_points=after-before))
            if scope == 'indist_core18':
                before, after = old['macro_balanced_acc'], current['macro_balanced_acc']
                comparison_rows.append(dict(scope=scope, condition=condition, metric='macro_balanced_acc',
                    reference_224=before, candidate_336=after, delta_percentage_points=after-before))
    comparison = pd.DataFrame(comparison_rows)
    comparison.to_csv(OUTPUT / 'comparison_224_vs_336.csv', index=False)
    print(comparison.to_string(index=False))
    conclusion = dict(schema='resolution336-comparison/v1', reference_summary_sha256=sha256_file(historical_path),
        current_package_sha256=PACKAGE_SHA, training_selection=gates['training_selection'], G72=summary['G72'],
        acceptance='requires_review_of_quality_and_pending_gates; no automatic replacement of 224 demo',
        comparison='descriptive post-hoc comparison; no paired significance claim',
        differences=['input 336 versus 224',
                     f"train microbatch{CONTRACT['train_batch']}/accum{CONTRACT['train_grad_accum_steps']} versus16/accum2; SRM BatchNorm may differ",
                     'resolution-specific baseline selection calibration',
                     f"inference batch{CONTRACT['inference_batch']} versus16; fresh calibration"])
    write_json(OUTPUT / 'comparison_conclusion.json', conclusion)

In [ ]:
# 10. Bàn giao chỉ bằng chứng của run này; không ghi đè run cũ hoặc weights/calibration.
with step('finalize'):
    required = {'git_source', 'environment_and_tests', 'catalog_preflight', 'all_archive_content_audit',
                'content_overlap', 'model_load', 'all_dataset_predictions', 'latency_and_gates', 'comparison_with_224'}
    previous = state['steps'][:-1]
    if not required.issubset({s['name'] for s in previous}) or any(s['status'] != 'completed' for s in previous):
        raise RuntimeError('Không thể finalize khi thiếu bước hoặc có bước thất bại. Tạo run mới.')
    if state.get('completed_jobs') != state.get('expected_jobs') or not state.get('completed_jobs'):
        raise RuntimeError('Chưa có đủ job đo thực tế.')
    db.commit()
    db.close()
    state['coverage'] = 'all eight configured archives audited and evaluated; no image sampling'
    state['interpretation'] = 'execution completion is not a PASS of all scientific/product gates'
    state['limits'] = ['post-hoc remeasurement, not blind test',
        'exact-content overlap checks only; perceptual/generator-disjoint audit not completed',
        'targeted tests only unless full_repository_tests_requested=true',
        'no localization ground truth, no forensic reliability certification',
        'CI by image does not cover all repeated-reference/generator dependencies']
persist('completed')
print('Đã hoàn tất thực thi theo phạm vi. Đọc gates.json và run_status.json; không mặc định mọi gate PASS.')

## Đọc kết quả và chốt mô hình

Đầu ra trên Drive: thư mục `audit_reports/master_audit_336_…` của chính run này.
Đọc `run_status.json`, `gates.json`, `benchmark_summary.json` và bảng CSV đo thực tế.
Chỉ `completed` khi đủ 37 job; trạng thái hoàn thành thực thi không đồng nghĩa mọi tiêu chí chất lượng đạt.

So sánh với bản 224 bằng cùng danh mục, split và chỉ số. Báo rõ khác biệt về độ phân giải,
microbatch và điểm vận hành đã hiệu chuẩn. Giữ bản 224 nếu 336 không có lợi ích được chứng minh.
Các kết quả CFE đã được xem trong nghiên cứu, nên đây là hậu kiểm, không phải blind test mới.
Các ảnh suy biến lưu sẵn giữ nguyên byte, sau đó được tiền xử lý lên 336; ghi rõ điều này khi phân tích.